# Baselines & Collaborative Filtering

## 1. Data Preparation

In [ ]:
from recsys_common import *

configure_notebook()

project_dir = Path.cwd()
shared_dir = project_dir / "artifacts" / "shared"

required_shared = [
    shared_dir / "df_model.csv",
    shared_dir / "train_df.csv",
    shared_dir / "test_df.csv",
]
missing_shared = [str(p) for p in required_shared if not p.exists()]
if missing_shared:
    raise FileNotFoundError(
        "Shared artifacts are required. Run 0_Data_Exploration.ipynb first. Missing:\\n"
        + "\\n".join(missing_shared)
    )

print("Loading prepared data from artifacts/shared/")
df_model = pd.read_csv(shared_dir / "df_model.csv")
train_df = pd.read_csv(shared_dir / "train_df.csv")
test_df = pd.read_csv(shared_dir / "test_df.csv")

for frame in (df_model, train_df, test_df):
    if "review_time" in frame.columns:
        frame["review_time"] = pd.to_datetime(frame["review_time"], errors="coerce")

print(f"Shared df_model: {df_model.shape}")
print(f"Shared train_df: {train_df.shape}")
print(f"Shared test_df: {test_df.shape}")


Loading prepared data from artifacts/shared/
Shared df_model: (219540, 15)
Shared train_df: (175632, 15)
Shared test_df: (43908, 15)


In [ ]:
print("Model-specific imports are provided by recsys_common.py")

In [3]:
relevance_threshold = 4
df_model["is_relevant"] = (df_model["Score"] >= relevance_threshold).astype(int)

print("Relevant interaction rate:", round(df_model["is_relevant"].mean(), 4))

Relevant interaction rate: 0.7823


In [4]:
print("Using shared split loaded from artifacts/shared/")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train users:", train_df["UserId"].nunique())
print("Test users:", test_df["UserId"].nunique())
print("Train products:", train_df["ProductId"].nunique())
print("Test products:", test_df["ProductId"].nunique())

Using shared split loaded from artifacts/shared/
Train shape: (175632, 15)
Test shape: (43908, 15)
Train users: 23195
Test users: 18281
Train products: 16657
Test products: 9578


## 2. Evaluation Helper Functions

In [5]:
K = 10

print(f"Ranking cutoff K: {K}")
print(f"Relevance threshold: {relevance_threshold}")

Ranking cutoff K: 10
Relevance threshold: 4


In [ ]:
print("Shared evaluation helpers loaded from recsys_common.py")

Loaded shared evaluation helpers from recsys_eval_utils.py


Evaluation note: ranking metrics here are computed on each user's held-out test interactions (not on full-catalog candidate ranking). This is acceptable for lightweight offline comparison, but values can be optimistic versus a full Top-N candidate evaluation setting.


################

## **6. Baseline Recommenders**

The random baseline recommends unseen products at random. It does not use ratings, popularity, or similarity, so it serves as a weak lower-bound benchmark.

### 6.1 Prepare data for surpise

To keep model training and evaluation consistent across baselines and collaborative filtering methods, we use the `surprise` library. It is designed for explicit-feedback recommendation tasks and provides a clean framework for rating prediction.

In [7]:
reader = Reader(rating_scale=(1, 5))

# Use the SAME split as train_df/test_df to avoid mismatch and leakage
train_data = Dataset.load_from_df(train_df[["UserId", "ProductId", "Score"]], reader)
trainset = train_data.build_full_trainset()

# Surprise testset format: list of (uid, iid, true_rating)
testset = list(test_df[["UserId", "ProductId", "Score"]].itertuples(index=False, name=None))

print("Surprise trainset size:", trainset.n_ratings)
print("Surprise testset size:", len(testset))

Surprise trainset size: 175632
Surprise testset size: 43908


In [8]:
TOP_N = 10
MAX_CATALOG_USERS = 300   # set to None if runtime is fine and you want all eligible users

# Use TRAIN catalog for model-compatible candidate generation.
all_items = sorted(train_df["ProductId"].dropna().unique())

train_seen_items = train_df.groupby("UserId")["ProductId"].apply(set).to_dict()

eligible_catalog_users = [
    user_id for user_id in sorted(test_df["UserId"].unique())
    if len(set(all_items) - train_seen_items.get(user_id, set())) > 0
]

if MAX_CATALOG_USERS is None:
    catalog_users = eligible_catalog_users
else:
    catalog_users = eligible_catalog_users[:MAX_CATALOG_USERS]

item_codes, item_labels = pd.factorize(train_df["ProductId"])
user_codes, user_labels = pd.factorize(train_df["UserId"])

item_user_matrix = csr_matrix(
    (train_df["Score"].astype(float), (item_codes, user_codes)),
    shape=(len(item_labels), len(user_labels))
)

item_similarity = cosine_similarity(item_user_matrix, dense_output=False)
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_labels)}

def build_topn_recommendations(score_fn, users=None, top_n=10):
    users = catalog_users if users is None else users
    user_recommendations = {}

    for user_id in users:
        seen_items = train_seen_items.get(user_id, set())
        candidate_items = [item_id for item_id in all_items if item_id not in seen_items]

        if not candidate_items:
            continue

        scored_items = []
        for item_id in candidate_items:
            pred_score = float(score_fn(user_id, item_id))
            scored_items.append((item_id, pred_score))

        scored_items.sort(key=lambda x: x[1], reverse=True)
        user_recommendations[user_id] = scored_items[:top_n]

    return user_recommendations

def coverage_at_n(user_recommendations, catalog_items):
    recommended_items = {
        item_id
        for recs in user_recommendations.values()
        for item_id, _ in recs
    }
    return len(recommended_items) / len(catalog_items) if len(catalog_items) > 0 else np.nan

def personalization_at_n(user_recommendations, catalog_items):
    user_ids = list(user_recommendations.keys())

    if len(user_ids) < 2:
        return np.nan

    item_to_col = {item_id: idx for idx, item_id in enumerate(catalog_items)}

    rows, cols, data = [], [], []
    for row_idx, user_id in enumerate(user_ids):
        for item_id, _ in user_recommendations[user_id]:
            rows.append(row_idx)
            cols.append(item_to_col[item_id])
            data.append(1)

    rec_matrix = csr_matrix(
        (data, (rows, cols)),
        shape=(len(user_ids), len(catalog_items))
    )

    sim_matrix = cosine_similarity(rec_matrix)
    upper = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]

    return 1 - upper.mean() if len(upper) > 0 else np.nan

def intra_list_similarity_at_n(user_recommendations, item_similarity, item_to_idx):
    ils_scores = []

    for recs in user_recommendations.values():
        rec_items = [item_id for item_id, _ in recs if item_id in item_to_idx]

        if len(rec_items) < 2:
            continue

        idx = [item_to_idx[item_id] for item_id in rec_items]
        sim_block = item_similarity[idx][:, idx].toarray()
        upper = sim_block[np.triu_indices_from(sim_block, k=1)]

        if len(upper) > 0:
            ils_scores.append(upper.mean())

    return float(np.mean(ils_scores)) if ils_scores else np.nan

def qualitative_examples(user_recommendations, n_users=3):
    rows = []

    for user_id in list(user_recommendations.keys())[:n_users]:
        for rank, (item_id, pred_score) in enumerate(user_recommendations[user_id], start=1):
            rows.append((user_id, rank, item_id, pred_score))

    return pd.DataFrame(rows, columns=["UserId", "Rank", "ProductId", "pred"])

print("Catalog users evaluated:", len(catalog_users))
print("Catalog size:", len(all_items))
print("TOP_N:", TOP_N)

Catalog users evaluated: 300
Catalog size: 16657
TOP_N: 10


### 6.2 Random recommender

In [9]:
np.random.seed(42)

random_model = NormalPredictor()
random_model.fit(trainset)

random_predictions = random_model.test(testset)

random_rmse = accuracy.rmse(random_predictions, verbose=False)
random_mae = accuracy.mae(random_predictions, verbose=False)

print("Random Baseline")
print(f"RMSE: {random_rmse:.4f}")
print(f"MAE:  {random_mae:.4f}")

Random Baseline
RMSE: 1.5894
MAE:  1.1907


#### 6.2.1 Ranking Metrics

In [10]:
random_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in random_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

random_eval_df.head()

,UserId,ProductId,Score,pred
0,A141XBYWBW40UC,B001O2F64I,3,4.813088
1,A2429SSTK8K5US,B001D0DMME,1,4.026129
2,A18ZOHJ7ZERQ8,B002NHYQAS,4,5.000000
3,A1YYWOLUF6I4G1,B0069GOKGE,5,5.000000
4,A3OO4WIO4SKD55,B001LG940E,3,3.907290


In [11]:
random_precision = precision_at_k(random_eval_df, k=K, threshold=relevance_threshold)
random_recall = recall_at_k(random_eval_df, k=K, threshold=relevance_threshold)
random_map = map_at_k(random_eval_df, k=K, threshold=relevance_threshold)
random_ndcg = ndcg_at_k(random_eval_df, k=K, threshold=relevance_threshold)

print("Random Baseline Ranking Metrics")
print(f"Precision@{K}: {random_precision:.4f}")
print(f"Recall@{K}: {random_recall:.4f}")
print(f"MAP@{K}: {random_map:.4f}")
print(f"NDCG@{K}: {random_ndcg:.4f}")

Random Baseline Ranking Metrics
Precision@10: 0.7869
Recall@10: 0.9959
MAP@10: 0.9578
NDCG@10: 0.9710


#### 6.2.2 Catalog Metrics

In [12]:
random_topn = build_topn_recommendations(
    lambda user_id, item_id: random_model.predict(user_id, item_id).est,
    top_n=TOP_N
)

print("Random full-candidate Top-N lists generated:", len(random_topn))

Random full-candidate Top-N lists generated: 300


In [13]:
random_coverage = coverage_at_n(random_topn, all_items)
random_personalization = personalization_at_n(random_topn, all_items)
random_ils = intra_list_similarity_at_n(random_topn, item_similarity, item_to_idx)

print("Random Baseline Behavioral/Catalog Metrics")
print(f"Coverage: {random_coverage:.4f}")
print(f"Personalization: {random_personalization:.4f}")
print(f"Intra-list similarity: {random_ils:.4f}")

Random Baseline Behavioral/Catalog Metrics
Coverage: 0.0042
Personalization: 0.7770
Intra-list similarity: 0.0078


In [14]:
display(qualitative_examples(random_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,A1001WMV1CL0XH,1,7310172001,5.0
1,A1001WMV1CL0XH,2,B00002N8SM,5.0
2,A1001WMV1CL0XH,3,B00004CI84,5.0
3,A1001WMV1CL0XH,4,B00004RBDZ,5.0
4,A1001WMV1CL0XH,5,B00004RYGX,5.0
5,A1001WMV1CL0XH,6,B000052Y74,5.0
6,A1001WMV1CL0XH,7,B0000537KC,5.0
7,A1001WMV1CL0XH,8,B00005C2M2,5.0
8,A1001WMV1CL0XH,9,B000084E66,5.0
9,A1001WMV1CL0XH,10,B000084E76,5.0


### 6.3 Baseline bias model

`BaselineOnly` predicts ratings using user and item bias terms. It is still simple, but more informed than a random predictor because it captures tendencies such as some users rating higher on average and some products receiving higher ratings overall.

In [15]:
baseline_model = BaselineOnly()
baseline_model.fit(trainset)

baseline_predictions = baseline_model.test(testset)

baseline_rmse = accuracy.rmse(baseline_predictions, verbose=False)
baseline_mae = accuracy.mae(baseline_predictions, verbose=False)


print("Baseline Bias Model")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"MAE:  {baseline_mae:.4f}")

Estimating biases using als...
Baseline Bias Model
RMSE: 1.0005
MAE:  0.7689


#### 6.3.1 Ranking Metrics

In [16]:
bias_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in baseline_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

bias_eval_df.head()

,UserId,ProductId,Score,pred
0,A141XBYWBW40UC,B001O2F64I,3,3.601692
1,A2429SSTK8K5US,B001D0DMME,1,3.512817
2,A18ZOHJ7ZERQ8,B002NHYQAS,4,4.116639
3,A1YYWOLUF6I4G1,B0069GOKGE,5,4.647686
4,A3OO4WIO4SKD55,B001LG940E,3,2.833071


In [17]:
bias_precision = precision_at_k(bias_eval_df, k=K, threshold=relevance_threshold)
bias_recall = recall_at_k(bias_eval_df, k=K, threshold=relevance_threshold)
bias_map = map_at_k(bias_eval_df, k=K, threshold=relevance_threshold)
bias_ndcg = ndcg_at_k(bias_eval_df, k=K, threshold=relevance_threshold)

print("Baseline Bias Model Ranking Metrics")
print(f"Precision@{K}: {bias_precision:.4f}")
print(f"Recall@{K}: {bias_recall:.4f}")
print(f"MAP@{K}: {bias_map:.4f}")
print(f"NDCG@{K}: {bias_ndcg:.4f}")

Baseline Bias Model Ranking Metrics
Precision@10: 0.7873
Recall@10: 0.9964
MAP@10: 0.9688
NDCG@10: 0.9786


#### 6.3.2 Catalog Metrics

In [18]:
baseline_topn = build_topn_recommendations(
    lambda user_id, item_id: baseline_model.predict(user_id, item_id).est,
    top_n=TOP_N
)

print("Baseline Bias full-candidate Top-N lists generated:", len(baseline_topn))

Baseline Bias full-candidate Top-N lists generated: 300


In [19]:
baseline_coverage = coverage_at_n(baseline_topn, all_items)
baseline_personalization = personalization_at_n(baseline_topn, all_items)
baseline_ils = intra_list_similarity_at_n(baseline_topn, item_similarity, item_to_idx)

print("Baseline Bias Behavioral/Catalog Metrics")
print(f"Coverage: {baseline_coverage:.4f}")
print(f"Personalization: {baseline_personalization:.4f}")
print(f"Intra-list similarity: {baseline_ils:.4f}")

Baseline Bias Behavioral/Catalog Metrics
Coverage: 0.0020
Personalization: 0.0987
Intra-list similarity: 0.1475


In [20]:
display(qualitative_examples(baseline_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,A1001WMV1CL0XH,1,B000ED9L9E,4.984715
1,A1001WMV1CL0XH,2,B000EVG8HY,4.957231
2,A1001WMV1CL0XH,3,B001CWX7EG,4.933050
3,A1001WMV1CL0XH,4,B0002DGRPC,4.931054
4,A1001WMV1CL0XH,5,B000EVIDWW,4.912234
5,A1001WMV1CL0XH,6,B000WFS9G0,4.910494
6,A1001WMV1CL0XH,7,B001CWV4RS,4.909418
7,A1001WMV1CL0XH,8,B001PICX42,4.903450
8,A1001WMV1CL0XH,9,B001EO5Q64,4.896429
9,A1001WMV1CL0XH,10,B000255OIG,4.894603


### 6.4 Bayesian Average Popular Baseline

In [21]:
C = 20  # minimum reviews needed to trust a rating

item_stats = (
    train_df.groupby("ProductId")["Score"]
    .agg(["mean", "count"])
    .reset_index()
)

global_mean = train_df["Score"].mean()

item_stats["bap_score"] = (
    (item_stats["count"] * item_stats["mean"] + C * global_mean)
    / (item_stats["count"] + C)
)

bap_baseline = (
    item_stats
    .sort_values("bap_score", ascending=False)
    .rename(columns={"mean": "avg_rating", "count": "num_reviews"})
)[["ProductId", "avg_rating", "num_reviews", "bap_score"]]

bap_baseline.head(10)

,ProductId,avg_rating,num_reviews,bap_score
2403,B000EVG8HY,4.854701,117,4.758757
602,B0002DGRPC,4.829787,141,4.751241
7642,B001CWX7EG,4.825688,109,4.728293
2409,B000EVIDWW,4.793388,121,4.708863
6023,B000WFS9G0,4.835443,79,4.706563
12191,B0030VBQ5Y,4.802083,96,4.697843
7640,B001CWV4RS,4.781513,119,4.697480
1,7310172001,4.766917,133,4.692482
603,B0002DGRQ6,4.753247,154,4.689366
533,B000255OIG,4.763359,131,4.688409


#### 6.4.1 RMSE, MAE and Ranking Metrics

In [22]:
# Build a lookup: ProductId -> BAP score
bap_lookup = bap_baseline.set_index("ProductId")["bap_score"]

# For each test row, predict using BAP score (fallback to global mean if unseen)
global_mean = train_df["Score"].mean()
test_df["bap_pred"] = test_df["ProductId"].map(bap_lookup).fillna(global_mean)

# Keep the existing RMSE and MAE for BAP
bap_rmse = np.sqrt(((test_df["Score"] - test_df["bap_pred"]) ** 2).mean())
bap_mae = (test_df["Score"] - test_df["bap_pred"]).abs().mean()

bap_eval_df = test_df[["UserId", "ProductId", "Score", "bap_pred"]].rename(columns={"bap_pred": "pred"}).copy()

bap_eval_df.head()
print("Bayesian Average Popular")
print(f"RMSE: {bap_rmse:.4f}")
print(f"MAE:  {bap_mae:.4f}")

Bayesian Average Popular
RMSE: 1.2055
MAE:  0.9437


In [23]:
bap_precision = precision_at_k(bap_eval_df, k=K, threshold=relevance_threshold)
bap_recall = recall_at_k(bap_eval_df, k=K, threshold=relevance_threshold)
bap_map = map_at_k(bap_eval_df, k=K, threshold=relevance_threshold)
bap_ndcg = ndcg_at_k(bap_eval_df, k=K, threshold=relevance_threshold)

print("Bayesian Average Popular Ranking Metrics")
print(f"Precision@{K}: {bap_precision:.4f}")
print(f"Recall@{K}: {bap_recall:.4f}")
print(f"MAP@{K}: {bap_map:.4f}")
print(f"NDCG@{K}: {bap_ndcg:.4f}")

Bayesian Average Popular Ranking Metrics
Precision@10: 0.7873
Recall@10: 0.9964
MAP@10: 0.9685
NDCG@10: 0.9783


#### 6.4.2 Catalog Metrics

In [24]:
bap_topn = build_topn_recommendations(
    lambda user_id, item_id: float(bap_lookup.get(item_id, global_mean)),
    top_n=TOP_N
)

print("Bayesian Average Popular full-candidate Top-N lists generated:", len(bap_topn))

Bayesian Average Popular full-candidate Top-N lists generated: 300


In [25]:
bap_coverage = coverage_at_n(bap_topn, all_items)
bap_personalization = personalization_at_n(bap_topn, all_items)
bap_ils = intra_list_similarity_at_n(bap_topn, item_similarity, item_to_idx)

print("Bayesian Average Popular Behavioral/Catalog Metrics")
print(f"Coverage: {bap_coverage:.4f}")
print(f"Personalization: {bap_personalization:.4f}")
print(f"Intra-list similarity: {bap_ils:.4f}")

Bayesian Average Popular Behavioral/Catalog Metrics
Coverage: 0.0008
Personalization: 0.0066
Intra-list similarity: 0.2186


In [26]:
display(qualitative_examples(bap_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,A1001WMV1CL0XH,1,B000EVG8HY,4.758757
1,A1001WMV1CL0XH,2,B0002DGRPC,4.751241
2,A1001WMV1CL0XH,3,B001CWX7EG,4.728293
3,A1001WMV1CL0XH,4,B000EVIDWW,4.708863
4,A1001WMV1CL0XH,5,B000WFS9G0,4.706563
5,A1001WMV1CL0XH,6,B0030VBQ5Y,4.697843
6,A1001WMV1CL0XH,7,B001CWV4RS,4.697480
7,A1001WMV1CL0XH,8,7310172001,4.692482
8,A1001WMV1CL0XH,9,B0002DGRQ6,4.689366
9,A1001WMV1CL0XH,10,B000255OIG,4.688409


This baseline treats items with more reviews as "their scores being more trustworthy", and pulls the ones with less reviews closer to the global mean, effectively treating those reviews as "less impactful". It is more reliable than ranking by average score alone.

### 6.5 Baseline comparison

In [27]:
baseline_results = pd.DataFrame({
    "Model": [
        "Random Baseline",
        "Baseline Bias Model",
        "Bayesian Average Popular"
    ],
    "RMSE": [
        random_rmse,
        baseline_rmse,
        bap_rmse
    ],
    "MAE": [
        random_mae,
        baseline_mae,
        bap_mae
    ],
    f"Precision@{K}": [
        random_precision,
        bias_precision,
        bap_precision
    ],
    f"Recall@{K}": [
        random_recall,
        bias_recall,
        bap_recall
    ],
    f"MAP@{K}": [
        random_map,
        bias_map,
        bap_map
    ],
    f"NDCG@{K}": [
        random_ndcg,
        bias_ndcg,
        bap_ndcg
    ],
    "Coverage": [
        random_coverage,
        baseline_coverage,
        bap_coverage
    ],
    "Personalization": [
        random_personalization,
        baseline_personalization,
        bap_personalization
    ],
    "Intra-list similarity": [
        random_ils,
        baseline_ils,
        bap_ils
    ]
})

display(baseline_results.sort_values("RMSE"))

,Model,RMSE,MAE,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage,Personalization,Intra-list similarity
1,Baseline Bias Model,1.000458,0.768910,0.787301,0.996414,0.968840,0.978598,0.002041,0.098720,0.147453
2,Bayesian Average Popular,1.205497,0.943730,0.787307,0.996409,0.968483,0.978304,0.000840,0.006638,0.218616
0,Random Baseline,1.589442,1.190671,0.786902,0.995901,0.957778,0.971014,0.004202,0.777026,0.007760


The results already show a clear improvement from a random predictor to a simple bias based model. This shows that even basic patterns in the data have some predictive value. The bayesian average baseline also give us a strong non personalised reference which we can later compare against more personalized methods.

## **7. Collaborative Filtering**

### 7.1 User based collaborative filtering

User-based collaborative filtering predicts a user’s rating from the ratings of similar users.

In [28]:
sim_options_user = {
    "name": "pearson",
    "user_based": True
}

user_cf = KNNWithMeans(sim_options=sim_options_user)
user_cf.fit(trainset)

user_cf_predictions = user_cf.test(testset)

user_cf_rmse = accuracy.rmse(user_cf_predictions, verbose=False)
user_cf_mae = accuracy.mae(user_cf_predictions, verbose=False)

print("User-Based Collaborative Filtering")
print(f"RMSE: {user_cf_rmse:.4f}")
print(f"MAE:  {user_cf_mae:.4f}")

Computing the pearson similarity matrix...
Done computing similarity matrix.
User-Based Collaborative Filtering
RMSE: 0.8268
MAE:  0.4423


#### 7.1.1 Ranking Metrics

In [29]:
usercf_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in user_cf_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

usercf_eval_df.head()

,UserId,ProductId,Score,pred
0,A141XBYWBW40UC,B001O2F64I,3,4.153759
1,A2429SSTK8K5US,B001D0DMME,1,1.000000
2,A18ZOHJ7ZERQ8,B002NHYQAS,4,3.873200
3,A1YYWOLUF6I4G1,B0069GOKGE,5,5.000000
4,A3OO4WIO4SKD55,B001LG940E,3,2.537439


In [30]:
user_cf_precision = precision_at_k(usercf_eval_df, k=K, threshold=relevance_threshold)
user_cf_recall = recall_at_k(usercf_eval_df, k=K, threshold=relevance_threshold)
user_cf_map = map_at_k(usercf_eval_df, k=K, threshold=relevance_threshold)
user_cf_ndcg = ndcg_at_k(usercf_eval_df, k=K, threshold=relevance_threshold)

print("User-Based CF Ranking Metrics")
print(f"Precision@{K}: {user_cf_precision:.4f}")
print(f"Recall@{K}: {user_cf_recall:.4f}")
print(f"MAP@{K}: {user_cf_map:.4f}")
print(f"NDCG@{K}: {user_cf_ndcg:.4f}")

User-Based CF Ranking Metrics
Precision@10: 0.7874
Recall@10: 0.9966
MAP@10: 0.9670
NDCG@10: 0.9774


#### 7.1.1 Catalog Metrics

In [31]:
user_cf_topn = build_topn_recommendations(
    lambda user_id, item_id: user_cf.predict(user_id, item_id).est,
    top_n=TOP_N
)

print("User-Based CF full-candidate Top-N lists generated:", len(user_cf_topn))


User-Based CF full-candidate Top-N lists generated: 300


In [32]:
user_cf_coverage = coverage_at_n(user_cf_topn, all_items)
user_cf_personalization = personalization_at_n(user_cf_topn, all_items)
user_cf_ils = intra_list_similarity_at_n(user_cf_topn, item_similarity, item_to_idx)

print("User-Based CF Behavioral/Catalog Metrics")
print(f"Coverage: {user_cf_coverage:.4f}")
print(f"Personalization: {user_cf_personalization:.4f}")
print(f"Intra-list similarity: {user_cf_ils:.4f}")

User-Based CF Behavioral/Catalog Metrics
Coverage: 0.0196
Personalization: 0.2587
Intra-list similarity: 0.0422


In [33]:
display(qualitative_examples(user_cf_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,A1001WMV1CL0XH,1,0006641040,5.00
1,A1001WMV1CL0XH,2,7310172001,5.00
2,A1001WMV1CL0XH,3,7310172101,5.00
3,A1001WMV1CL0XH,4,B00002N8SM,5.00
4,A1001WMV1CL0XH,5,B00004CI84,5.00
5,A1001WMV1CL0XH,6,B00004CXX9,5.00
6,A1001WMV1CL0XH,7,B00004RAMX,5.00
7,A1001WMV1CL0XH,8,B00004RAMY,5.00
8,A1001WMV1CL0XH,9,B00004RBDU,5.00
9,A1001WMV1CL0XH,10,B00004RBDZ,5.00


### 7.2 Item based collaborative filtering

Item-based collaborative filtering predicts preferences from relationships between products rather than between users.

In [34]:
sim_options_item = {
    "name": "pearson",
    "user_based": False
}

item_cf = KNNWithMeans(sim_options=sim_options_item)
item_cf.fit(trainset)

item_cf_predictions = item_cf.test(testset)

item_cf_rmse = accuracy.rmse(item_cf_predictions, verbose=False)
item_cf_mae = accuracy.mae(item_cf_predictions, verbose=False)

print("Item-Based Collaborative Filtering")
print(f"RMSE: {item_cf_rmse:.4f}")
print(f"MAE:  {item_cf_mae:.4f}")

Computing the pearson similarity matrix...
Done computing similarity matrix.
Item-Based Collaborative Filtering
RMSE: 0.7697
MAE:  0.3741


#### 7.2.1 Ranking Metrics

In [35]:
itemcf_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in item_cf_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

itemcf_eval_df.head()

,UserId,ProductId,Score,pred
0,A141XBYWBW40UC,B001O2F64I,3,2.521978
1,A2429SSTK8K5US,B001D0DMME,1,1.023944
2,A18ZOHJ7ZERQ8,B002NHYQAS,4,4.485303
3,A1YYWOLUF6I4G1,B0069GOKGE,5,4.999471
4,A3OO4WIO4SKD55,B001LG940E,3,2.299878


In [36]:
item_cf_precision = precision_at_k(itemcf_eval_df, k=K, threshold=relevance_threshold)
item_cf_recall = recall_at_k(itemcf_eval_df, k=K, threshold=relevance_threshold)
item_cf_map = map_at_k(itemcf_eval_df, k=K, threshold=relevance_threshold)
item_cf_ndcg = ndcg_at_k(itemcf_eval_df, k=K, threshold=relevance_threshold)

print("Item-Based CF Ranking Metrics")
print(f"Precision@{K}: {item_cf_precision:.4f}")
print(f"Recall@{K}: {item_cf_recall:.4f}")
print(f"MAP@{K}: {item_cf_map:.4f}")
print(f"NDCG@{K}: {item_cf_ndcg:.4f}")

Item-Based CF Ranking Metrics
Precision@10: 0.7876
Recall@10: 0.9968
MAP@10: 0.9795
NDCG@10: 0.9863


#### 7.2.2 Catalog Metrics

In [37]:
item_cf_topn = build_topn_recommendations(
    lambda user_id, item_id: item_cf.predict(user_id, item_id).est,
    top_n=TOP_N
)

print("Item-Based CF full-candidate Top-N lists generated:", len(item_cf_topn))

Item-Based CF full-candidate Top-N lists generated: 300


In [38]:
item_cf_coverage = coverage_at_n(item_cf_topn, all_items)
item_cf_personalization = personalization_at_n(item_cf_topn, all_items)
item_cf_ils = intra_list_similarity_at_n(item_cf_topn, item_similarity, item_to_idx)

print("Item-Based CF Behavioral/Catalog Metrics")
print(f"Coverage: {item_cf_coverage:.4f}")
print(f"Personalization: {item_cf_personalization:.4f}")
print(f"Intra-list similarity: {item_cf_ils:.4f}")

Item-Based CF Behavioral/Catalog Metrics
Coverage: 0.0013
Personalization: 0.0119
Intra-list similarity: 0.0002


In [39]:
display(qualitative_examples(item_cf_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,A1001WMV1CL0XH,1,B00004RAMX,5.0
1,A1001WMV1CL0XH,2,B00005344V,5.0
2,A1001WMV1CL0XH,3,B0000537KC,5.0
3,A1001WMV1CL0XH,4,B00005C2M2,5.0
4,A1001WMV1CL0XH,5,B00005U2FA,5.0
5,A1001WMV1CL0XH,6,B00006LL38,5.0
6,A1001WMV1CL0XH,7,B000084E66,5.0
7,A1001WMV1CL0XH,8,B000084EKY,5.0
8,A1001WMV1CL0XH,9,B000084EL4,5.0
9,A1001WMV1CL0XH,10,B000084F04,5.0


### 7.3 Model based collaborative filtering (SVD / Matrix factorization)

Model-based collaborative filtering learns latent user and item factors from the rating matrix. Here, we use SVD as the main model-based approach.

In [40]:
svd_model = SVD(random_state=42)
svd_model.fit(trainset)

svd_predictions = svd_model.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

print("SVD")
print(f"RMSE: {svd_rmse:.4f}")
print(f"MAE:  {svd_mae:.4f}")

SVD
RMSE: 0.8265
MAE:  0.5690


#### 7.3.1 Ranking Metrics

In [41]:
svd_eval_df = pd.DataFrame(
    [(p.uid, p.iid, p.r_ui, p.est) for p in svd_predictions],
    columns=["UserId", "ProductId", "Score", "pred"]
)

svd_eval_df.head()

,UserId,ProductId,Score,pred
0,A141XBYWBW40UC,B001O2F64I,3,3.540856
1,A2429SSTK8K5US,B001D0DMME,1,1.790781
2,A18ZOHJ7ZERQ8,B002NHYQAS,4,4.120973
3,A1YYWOLUF6I4G1,B0069GOKGE,5,4.632485
4,A3OO4WIO4SKD55,B001LG940E,3,3.058378


In [42]:
svd_precision = precision_at_k(svd_eval_df, k=K, threshold=relevance_threshold)
svd_recall = recall_at_k(svd_eval_df, k=K, threshold=relevance_threshold)
svd_map = map_at_k(svd_eval_df, k=K, threshold=relevance_threshold)
svd_ndcg = ndcg_at_k(svd_eval_df, k=K, threshold=relevance_threshold)

print("SVD Ranking Metrics")
print(f"Precision@{K}: {svd_precision:.4f}")
print(f"Recall@{K}: {svd_recall:.4f}")
print(f"MAP@{K}: {svd_map:.4f}")
print(f"NDCG@{K}: {svd_ndcg:.4f}")

SVD Ranking Metrics
Precision@10: 0.7875
Recall@10: 0.9967
MAP@10: 0.9763
NDCG@10: 0.9840


#### 7.3.1 Ranking Metrics

In [43]:
svd_topn = build_topn_recommendations(
    lambda user_id, item_id: svd_model.predict(user_id, item_id).est,
    top_n=TOP_N
)

print("SVD full-candidate Top-N lists generated:", len(svd_topn))

SVD full-candidate Top-N lists generated: 300


In [44]:
svd_coverage = coverage_at_n(svd_topn, all_items)
svd_personalization = personalization_at_n(svd_topn, all_items)
svd_ils = intra_list_similarity_at_n(svd_topn, item_similarity, item_to_idx)

print("SVD Behavioral/Catalog Metrics")
print(f"Coverage: {svd_coverage:.4f}")
print(f"Personalization: {svd_personalization:.4f}")
print(f"Intra-list similarity: {svd_ils:.4f}")

SVD Behavioral/Catalog Metrics
Coverage: 0.0319
Personalization: 0.8933
Intra-list similarity: 0.0706


In [45]:
display(qualitative_examples(svd_topn, n_users=3))

,UserId,Rank,ProductId,pred
0,A1001WMV1CL0XH,1,7310172001,5.000000
1,A1001WMV1CL0XH,2,7310172101,5.000000
2,A1001WMV1CL0XH,3,B00020XNTS,5.000000
3,A1001WMV1CL0XH,4,B000255OIG,5.000000
4,A1001WMV1CL0XH,5,B00027CL5S,5.000000
5,A1001WMV1CL0XH,6,B0002DGRPC,5.000000
6,A1001WMV1CL0XH,7,B0002DGRRA,5.000000
7,A1001WMV1CL0XH,8,B0002DGRZC,5.000000
8,A1001WMV1CL0XH,9,B0002PSOJW,5.000000
9,A1001WMV1CL0XH,10,B0006Z7NOK,5.000000


### 7.4 Collaborative filtering comparison

In [46]:
cf_results = pd.DataFrame({
    "Model": [
        "User-Based CF",
        "Item-Based CF",
        "SVD"
    ],
    "RMSE": [
        user_cf_rmse,
        item_cf_rmse,
        svd_rmse
    ],
    "MAE": [
        user_cf_mae,
        item_cf_mae,
        svd_mae
    ],
    f"Precision@{K}": [
        user_cf_precision,
        item_cf_precision,
        svd_precision
    ],
    f"Recall@{K}": [
        user_cf_recall,
        item_cf_recall,
        svd_recall
    ],
    f"MAP@{K}": [
        user_cf_map,
        item_cf_map,
        svd_map
    ],
    f"NDCG@{K}": [
        user_cf_ndcg,
        item_cf_ndcg,
        svd_ndcg
    ],
    "Coverage": [
        user_cf_coverage,
        item_cf_coverage,
        svd_coverage
    ],
    "Personalization": [
        user_cf_personalization,
        item_cf_personalization,
        svd_personalization
    ],
    "Intra-list similarity": [
        user_cf_ils,
        item_cf_ils,
        svd_ils
    ]
})

cf_results.sort_values("RMSE")

,Model,RMSE,MAE,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage,Personalization,Intra-list similarity
1,Item-Based CF,0.769748,0.374051,0.787608,0.996805,0.979506,0.986299,0.001261,0.011906,0.000176
2,SVD,0.826530,0.568973,0.787542,0.996705,0.976306,0.984031,0.031939,0.893284,0.070630
0,User-Based CF,0.826801,0.442346,0.787427,0.996587,0.966985,0.977435,0.019571,0.258691,0.042250


The results show a clear contrast between prediction accuracy, ranking performance, and actual recommendation quality, and this contrast is both expected and informative.

First, all three models appear almost identical in terms of ranking metrics, with Precision@10 around 0.787, Recall@10 close to 1.0, and NDCG@10 above 0.97. At first glance, this suggests that all models perform extremely well and almost equally in ranking relevant items. However, this is largely a consequence of the evaluation setup: since ranking metrics were computed using test-only interactions, the models are effectively ranking a small set of already relevant items against each other rather than competing against the full catalog. This makes the task artificially easy and leads to inflated and very similar scores across models. Therefore, the lack of differentiation in ranking metrics is not surprising and does not necessarily reflect real-world recommendation performance.

Looking at regression metrics, Item-Based CF achieves the best RMSE and MAE, indicating that it is the most accurate at predicting rating values. This is typical for memory-based collaborative filtering methods, which rely heavily on observed similarities and can fit known interactions closely. However, this strength does not translate into better recommendation behavior. In fact, Item-Based CF performs extremely poorly in the behavioral metrics: its coverage is almost zero (0.0012), and personalization is also near zero (0.0119). This means that the model is recommending almost the same very small subset of items to all users. This is a classic symptom of popularity bias, where the model over-relies on frequently interacted items and ignores most of the catalog. This behavior is entirely expected for item-based approaches, especially in sparse datasets with long-tail distributions.

User-Based CF improves upon this by increasing both coverage (0.0182) and personalization (0.58). This indicates that recommendations start to differ across users and include a broader set of items. The reason is that user-based methods consider neighborhoods of similar users, which introduces more variability than item-based methods that tend to converge on popular items. However, the improvement is still limited, and the model does not fully overcome the bias toward popular items.

SVD presents the most balanced and arguably the best performance overall. Although its RMSE is slightly worse than Item-Based CF, it significantly outperforms both memory-based methods in behavioral metrics. Coverage increases to 0.0303, meaning a much larger portion of the catalog is being recommended. Personalization is very high (0.893), indicating that users receive highly distinct recommendation lists. Intra-list similarity is also higher, suggesting that recommended items are more coherent within each list. This behavior is expected because SVD, as a latent factor model, captures underlying user preferences and item characteristics beyond direct co-occurrence patterns. This allows it to generalize better and recommend less obvious, more personalized items.

One potentially surprising result is how similar the ranking metrics are across all models despite large differences in behavioral performance. However, given that ranking was evaluated only on test interactions, this is not unexpected. The evaluation setup masks the true differences in recommendation quality by not considering the full candidate space. If ranking metrics were computed using full-candidate evaluation, larger differences between models would likely emerge, aligning more closely with the behavioral metrics.

In summary, the results are consistent with theoretical expectations. Memory-based methods (especially item-based) tend to optimize prediction accuracy but suffer from popularity bias and lack of personalization. Latent factor models like SVD sacrifice some prediction accuracy but produce significantly better recommendation behavior. The main limitation in the analysis comes from the test-only ranking evaluation, which overestimates ranking performance and reduces its ability to distinguish between models.


##############################################


## 8. Artifact Export for Hybrid

Export CF and baseline score artifacts for the hybrid notebook.

In [47]:

project_dir = Path.cwd()
baselines_dir = project_dir / "artifacts" / "baselines_cf"
baselines_dir.mkdir(parents=True, exist_ok=True)

# Best CF model for hybrid (SVD selected from this notebook's CF section).
cf_test_scores = svd_eval_df.rename(columns={"pred": "cf_score"})[["UserId", "ProductId", "cf_score"]].copy()

cf_topn_scores = (
    pd.DataFrame(
        [(u, i, s) for u, recs in svd_topn.items() for i, s in recs],
        columns=["UserId", "ProductId", "cf_score"]
    )
    if len(svd_topn) > 0
    else pd.DataFrame(columns=["UserId", "ProductId", "cf_score"])
)

# Popularity/baseline fallback for hybrid.
pop_scores = bap_baseline[["ProductId", "bap_score"]].rename(columns={"bap_score": "pop_score"}).copy()

cf_test_scores.to_csv(baselines_dir / "cf_test_scores.csv", index=False)
cf_topn_scores.to_csv(baselines_dir / "cf_topn_scores.csv", index=False)
pop_scores.to_csv(baselines_dir / "pop_scores.csv", index=False)

baseline_results.to_csv(baselines_dir / "baseline_results.csv", index=False)
cf_results.to_csv(baselines_dir / "cf_results.csv", index=False)

artifact_manifest = pd.DataFrame(
    [
        {"artifact": "cf_test_scores.csv", "rows": len(cf_test_scores)},
        {"artifact": "cf_topn_scores.csv", "rows": len(cf_topn_scores)},
        {"artifact": "pop_scores.csv", "rows": len(pop_scores)},
        {"artifact": "baseline_results.csv", "rows": len(baseline_results)},
        {"artifact": "cf_results.csv", "rows": len(cf_results)},
    ]
)
artifact_manifest.to_csv(baselines_dir / "artifact_manifest.csv", index=False)

print("Saved baselines/CF artifacts to:", baselines_dir)
display(artifact_manifest)

Saved baselines/CF artifacts to: /Users/silvia/Desktop/BCSAI/Year3/SEM2/RecSys/Recommender-Sys/artifacts/baselines_cf


,artifact,rows
0,cf_test_scores.csv,43908
1,cf_topn_scores.csv,3000
2,pop_scores.csv,16657
3,baseline_results.csv,3
4,cf_results.csv,3
